In [1]:
# creating spark session
exec(open('/home/jovyan/.ipython/profile_default/startup/00-spark-session.py').read())

Spark 3.5.0 session ready as `spark` (Delta Lake enabled).


# Team Scores Comparison

## Difficulty
Medium

## Topics
- PySpark
- SQL
- Aggregation
- Self-Join
- UNION ALL
- Group By
- Arithmetic Operations
- Sorting

## Problem Statement

You are given a PySpark DataFrame named `matches` containing football match results.

### Dataset: `matches`

| Column | Data Type | Description |
|---|---|---|
| `match_id` | Integer | Unique identifier for each match |
| `home_team` | String | Name of the home team |
| `away_team` | String | Name of the away team |
| `home_score` | Integer | Goals scored by the home team |
| `away_score` | Integer | Goals scored by the away team |

## Task

Calculate the total goals **scored** and **conceded** by each team across all matches.

A team can appear in a match as either the **home team** or the **away team**.

### When a team plays at home

- Goals scored = `home_score`
- Goals conceded = `away_score`

### When a team plays away

- Goals scored = `away_score`
- Goals conceded = `home_score`

After calculating the totals, calculate:

`goal_difference = goals_scored - goals_conceded`

## Requirements

1. Calculate goals scored and conceded for teams playing at home.
2. Calculate goals scored and conceded for teams playing away.
3. Combine the home and away records.
4. Group the combined data by team.
5. Calculate:
   - `goals_scored`
   - `goals_conceded`
   - `goal_difference`
6. Return:
   - `team`
   - `goals_scored`
   - `goals_conceded`
   - `goal_difference`
7. Sort the result by:
   - `goal_difference` descending
   - `team` ascending for ties

## Expected Output

| team | goals_scored | goals_conceded | goal_difference |
|---|---:|---:|---:|
| Arsenal | 7 | 3 | 4 |
| Liverpool | 7 | 4 | 3 |
| Chelsea | 3 | 10 | -7 |

In [2]:
from pyspark.sql.types import (
    StructType,
    StructField,
    IntegerType,
    StringType
)

matches_data = [
    (1, "Arsenal",  "Chelsea",   3, 1),
    (2, "Chelsea",  "Liverpool", 2, 2),
    (3, "Liverpool", "Arsenal",  1, 0),
    (4, "Arsenal",  "Liverpool", 2, 1),
    (5, "Chelsea",  "Arsenal",   0, 2),
    (6, "Liverpool", "Chelsea",  3, 0)
]

matches_schema = StructType([
    StructField("match_id", IntegerType(), False),
    StructField("home_team", StringType(), False),
    StructField("away_team", StringType(), False),
    StructField("home_score", IntegerType(), False),
    StructField("away_score", IntegerType(), False)
])

matches = spark.createDataFrame(
    matches_data,
    matches_schema
)

matches.show()
matches.printSchema()

+--------+---------+---------+----------+----------+
|match_id|home_team|away_team|home_score|away_score|
+--------+---------+---------+----------+----------+
|       1|  Arsenal|  Chelsea|         3|         1|
|       2|  Chelsea|Liverpool|         2|         2|
|       3|Liverpool|  Arsenal|         1|         0|
|       4|  Arsenal|Liverpool|         2|         1|
|       5|  Chelsea|  Arsenal|         0|         2|
|       6|Liverpool|  Chelsea|         3|         0|
+--------+---------+---------+----------+----------+

root
 |-- match_id: integer (nullable = false)
 |-- home_team: string (nullable = false)
 |-- away_team: string (nullable = false)
 |-- home_score: integer (nullable = false)
 |-- away_score: integer (nullable = false)



In [3]:
from pyspark.sql.functions import *

# using SQL

In [4]:
matches.createOrReplaceTempView("matches")

In [29]:

spark.sql("""

with cte as  
        (SELECT 
            home_team as team_name,
            sum(home_score) as goals_scored,
            sum(away_score) as goals_conceded 
            from matches
            group by home_team
        UNION ALL
        SELECT 
        away_team as team_name,
        sum(away_score)  as goal_scored, 
        sum(home_score) as goals_conceded 
        from matches 
        group by away_team
)
,
    cte2 as  
    (
    SELECT team_name,
    sum(goals_scored) as goals_scored ,
    sum(goals_conceded) as goals_conceded 
    from cte 
    group by team_name
    )

SELECT *, 
(goals_scored - goals_conceded) as goal_difference 
from cte2 
ORDER BY
goal_difference DESC,
team_name ASC


    """).show()



+---------+------------+--------------+---------------+
|team_name|goals_scored|goals_conceded|goal_difference|
+---------+------------+--------------+---------------+
|  Arsenal|           7|             3|              4|
|Liverpool|           7|             4|              3|
|  Chelsea|           3|            10|             -7|
+---------+------------+--------------+---------------+

